# DINOv3 PV: manual masks reserved for final evaluation

Fresh RGB-only pilot: 120 images, 84 train / 36 final test. Frozen DINOv3 + frozen two-view InternVL. Partial machine pseudo-label losses, seven controls and three seeds. Final GT access happens in a separate process after prediction/checkpoint hashes are sealed.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle before running.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN loaded from Kaggle Secrets')
except Exception:
    print('HF_TOKEN secret not available; public or mounted model paths will be used')

os.environ['HF_HOME'] = '/kaggle/temp/model_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/kaggle/temp/model_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/temp/model_cache'

dino_weights = [p for p in Path('/kaggle/input').rglob('model.safetensors') if 'timmvit-large-patch16-dinov3-sat493m' in p.as_posix()]
assert len(dino_weights) == 1, f'Expected one mounted DINOv3 SAT checkpoint, found: {dino_weights}'
os.environ['DINOV3_MODEL_PATH'] = str(dino_weights[0])
print('DINOv3 cloud checkpoint:', dino_weights[0])


In [ ]:
repo = Path('/kaggle/temp/DINOV3')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Zh0416/DINOV3.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo), '--no-deps', '--no-build-isolation'], check=True)
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())
config = str(repo / 'configs' / 'kaggle_label_free.yaml')


## Fixed image-only protocol

No annotation inspection or historical supervised checkpoint is used. The historical manifest supplies exclusion RGB filenames only. Epochs and thresholds are fixed in the config.

In [ ]:
for stage in ['prepare', 'features', 'teacher', 'pseudo', 'train', 'seal']:
    print(f'\n=== LABEL-FREE STAGE: {stage} ===', flush=True)
    subprocess.run([sys.executable, '-u', str(repo / 'scripts' / '08_label_free_pipeline.py'), '--config', config, '--stage', stage], check=True)


## Final evaluation: first annotation access

All model choices and predictions are frozen. This process checks their hashes, then reads GT to report metrics and audit teacher/pseudo-label reliability. Do not adjust this protocol from its final test.

In [ ]:
subprocess.run([sys.executable, '-u', str(repo / 'scripts' / '09_final_evaluation.py'), '--output-root', '/kaggle/working/outputs'], check=True)
report = json.loads(Path('/kaggle/working/outputs/reports/label_free_final.json').read_text(encoding='utf-8'))
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))
